In [9]:
!pip install azure-eventhub

StatementMeta(, 5addb62e-dbb0-4c5f-b3d4-2433e2ab0334, 11, Finished, Available, Finished)

In [ ]:
import pandas as pd
import random
from datetime import datetime

# Master Product Map: ProductID → (ProductName, ProductCategory)
product_master = {
    "PROD0001": ("Pale Meadow", "Paint Shades"),
    "PROD0002": ("Tranquil Lavender", "Paint Shades"),
    "PROD0003": ("Whispering Blue", "Paint Shades"),
    "PROD0004": ("Whispering Blush", "Paint Shades"),
    "PROD0005": ("Ocean Mist", "Paint Shades"),
    "PROD0006": ("Sunset Coral", "Paint Shades"),
    "PROD0007": ("Forest Whisper", "Paint Shades"),
    "PROD0008": ("Morning Dew", "Paint Shades"),
    "PROD0009": ("Dusty Rose", "Paint Shades"),
    "PROD0010": ("Sage Harmony", "Paint Shades"),
    "PROD0011": ("Vanilla Dream", "Paint Shades"),
    "PROD0012": ("Charcoal Storm", "Paint Shades"),
    "PROD0013": ("Golden Wheat", "Paint Shades"),
    "PROD0014": ("Soft Pebble", "Paint Shades"),
    "PROD0015": ("Misty Gray", "Paint Shades"),
    "PROD0016": ("Rustic Clay", "Paint Shades"),
    "PROD0017": ("Ivory Pearl", "Paint Shades"),
    "PROD0018": ("Deep Forest", "Paint Shades"),
    "PROD0019": ("Autumn Spice", "Paint Shades"),
    "PROD0020": ("Coastal Whisper", "Paint Shades"),
    "PROD0021": ("Effervescent Jade", "Paint Shades"),
    "PROD0022": ("Frosted Blue", "Paint Shades"),
    "PROD0023": ("Frosted Lemon", "Paint Shades"),
    "PROD0024": ("Honeydew Sunrise", "Paint Shades"),
    "PROD0025": ("Lavender Whisper", "Paint Shades"),
    "PROD0026": ("Lilac Mist", "Paint Shades"),
    "PROD0027": ("Soft Creamsicle", "Paint Shades"),
    "PROD0028": ("Whispering Blush", "Paint Shades"),
    "PROD0029": ("Lavender Whisper", "Paint Shades"),
    "PROD0030": ("Lilac Mist", "Paint Shades"),
    "PROD0031": ("Soft Creamsicle", "Paint Shades"),
    "PROD0032": ("Whispering Blush", "Paint Shades"),
    "PROD0033": ("Cordless Airless Paint Sprayer", "Paint Sprayer"),
    "PROD0034": ("EZ-Coat Paint Sprayer", "Paint Sprayer"),
    "PROD0035": ("Electric Paint Sprayer 350", "Paint Sprayer"),
    "PROD0036": ("CB - Crimson Paint Sprayer", "Paint Sprayer"),
    "PROD0037": ("YP - Coat Paint Sprayer0", "Paint Sprayer"),
    "PROD0038": ("Handheld HVLP Paint Sprayer", "Paint Sprayer"),
    "PROD0039": ("Paint Safe Drop Cloth", "Paint Accessories"),
    "PROD0040": ("Paint Guard Reusable Drop Cloth", "Paint Accessories"),
    "PROD0041": ("Fine Finish Paint Brush", "Paint Accessories"),
    "PROD0042": ("All-Purpose Wall Paint Brush", "Paint Accessories"),
    "PROD0043": ("Large Area Applicator Brush", "Paint Accessories"),
    "PROD0044": ("Classic Flat Sash Brush", "Paint Accessories"),
    "PROD0045": ("Standard Paint Tray", "Paint Accessories"),
    "PROD0046": ("Deep Well Paint Tray", "Paint Accessories"),
    "PROD0047": ("Compact Paint Tray", "Paint Accessories"),
    "PROD0048": ("Heavy-Duty Paint Tray with Grid", "Paint Accessories"),
    "PROD0049": ("Blue Painter's Tape", "Paint Accessories"),
    "PROD0050": ("Green Painter's Tape", "Paint Accessories"),
    "PROD0051": ("Standard Paint Roller", "Paint Accessories"),
    "PROD0052": ("Ergonomic Grip Paint Roller", "Paint Accessories"),
    "PROD0053": ("Classic Wood Handle Paint Roller", "Paint Accessories"),
    "PROD0054": ("Wooden Handle Paint Roller", "Paint Accessories"),
}

store_locations = ["Los Angeles", "Seattle", "Ontario", "San Francisco", "Mountain View"]


forced_out_of_stock = {
    "Cordless Airless Paint Sprayer": "PROD0033",
    "EZ-Coat Paint Sprayer": "PROD0034",  
    "Electric Paint Sprayer 350": "PROD0035"
}

product_master["PROD0034"] = ("EZ-Coat Paint Sprayer", "Paint Sprayer")


def generate_inventory_and_status(product_id):
    if product_id in forced_out_of_stock.values():
        return 0, "Out of Stock"
    inv = random.randint(0, 100)
    if inv == 0:
        return inv, "Out of Stock"
    elif inv <= 15:
        return inv, "Low Stock"
    elif inv <= 50:
        return inv, "Medium Stock"
    else:
        return inv, "In Stock"

timestamp = datetime.now().strftime("%d-%m-%Y %H:%M")


records = []
used_ids = set(forced_out_of_stock.values())


for name, pid in forced_out_of_stock.items():
    pname, pcat = product_master[pid]
    records.append({
        "ProductID": pid,
        "ProductName": pname,
        "ProductCategory": pcat,
        "StoreLocation": random.choice(store_locations),
        "InventorySKU": 0,
        "StockAvailabilityStatus": "Out of Stock",
        "LastRestockDateTime": timestamp
    })


remaining_ids = [pid for pid in product_master if pid not in used_ids]
random.shuffle(remaining_ids)

for pid in remaining_ids:
    if len(records) >= 49:
        break
    pname, pcat = product_master[pid]
    inv, status = generate_inventory_and_status(pid)
    records.append({
        "ProductID": pid,
        "ProductName": pname,
        "ProductCategory": pcat,
        "StoreLocation": random.choice(store_locations),
        "InventorySKU": inv,
        "StockAvailabilityStatus": status,
        "LastRestockDateTime": timestamp
    })
    used_ids.add(pid)

# Convert to DataFrame
df_inventory = pd.DataFrame(records)
display(df_inventory)
spark_df = spark.createDataFrame(df_inventory)


StatementMeta(, 5addb62e-dbb0-4c5f-b3d4-2433e2ab0334, 12, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 4883deea-ff72-4a96-a318-c654633d8366)

In [ ]:


from azure.eventhub import EventHubProducerClient, EventData

# Event Hub configuration
CONNECTION_STR = "#Connection string Primary Key#"
EVENT_HUB_NAME = "#Eventhub Name#"

producer = EventHubProducerClient.from_connection_string(conn_str=CONNECTION_STR, eventhub_name=EVENT_HUB_NAME)

event_data_batch = producer.create_batch()
for row in spark_df.collect():
    event_data_batch.add(EventData(str(row.asDict())))

producer.send_batch(event_data_batch)
producer.close()
print("✅ Data also sent to Eventstream endpoint.")

StatementMeta(, 5addb62e-dbb0-4c5f-b3d4-2433e2ab0334, 14, Finished, Available, Finished)

✅ Data also sent to Eventstream endpoint.
